# Análise comparativa de representações
**Fernando Amorim Pontes Neto**

BoW, TF-IDF e dois modelos multilíngues no mesmo corpus. Execute todas as células a partir desta pasta. Os modelos são baixados no primeiro uso e a inferência é local.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.metrics.pairwise import cosine_similarity
from ouvidoria_core import *
base = carregar_base('manifestacoes.json')
assert len(base) == 40, 'A análise do desafio requer exatamente 40 registros.'
textos = base.texto.tolist()
ids = base.id.tolist()
saida = Path('resultados'); saida.mkdir(exist_ok=True)
print(f'Base: {len(base)} registros; textos >500 caracteres: {(base.texto.str.len()>500).sum()}')
if Path('PROVENIENCIA.json').exists():
    display(Markdown('**Proveniência:** ' + json.loads(Path('PROVENIENCIA.json').read_text())['descricao']))
display(base.head())

/home/user/ouvidoria/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Base: 40 registros; textos >500 caracteres: 5


**Proveniência:** Base demonstrativa sintética criada para o exercício, com autorização do usuário. Não corresponde a manifestações reais nem à base original do professor. Os resultados e o gabarito se aplicam apenas a estes 40 exemplos.

,id,data,categoria_oficial,texto
0,M001,2026-09-01,infraestrutura,"A calçada da Rua das Flores, perto do número 1..."
1,M002,2026-09-01,educação,A Escola Municipal Esperança está sem professo...
2,M003,2026-09-02,infraestrutura,"Há um buraco enorme na Av. Brasil, a avenida p..."
3,M004,2026-09-02,segurança,Moradores relatam furtos de bicicletas junto a...
4,M005,2026-09-03,meio ambiente,Uma pilha de pneus usados foi abandonada no te...


## Representações e pares obrigatórios
BoW e TF-IDF usam a mesma tokenização, minúsculas e remoção de acentos. Mantemos stopwords nesta comparação controlada para evidenciar também a sobreposição de palavras funcionais. O TF-IDF usa IDF suavizada e normalização L2. Nenhum modelo é treinado com os rótulos de categoria. Textos além do limite de tokens são agregados por média de janelas normalizadas, evitando truncamento silencioso.

In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
bow = CountVectorizer(strip_accents='unicode', lowercase=True)
tfidf = TfidfVectorizer(strip_accents='unicode', lowercase=True)
representacoes = {'BoW': bow.fit_transform(textos), 'TF-IDF': tfidf.fit_transform(textos)}
for nome in MODELOS:
    representacoes[nome.split('/')[-1]] = codificar(textos, nome)
pares = [('M003','M017'), ('M008','M022'), ('M008','M031')]
pos = {ident:i for i,ident in enumerate(ids)}
assert all(a in pos and b in pos for a,b in pares), 'Faltam IDs obrigatórios.'
linhas = []
for a,b in pares:
    r = {'Par': f'{a} × {b}'}
    for nome,x in representacoes.items():
        r[nome] = float(cosine_similarity(x[pos[a]:pos[a]+1],x[pos[b]:pos[b]+1])[0,0])
    linhas.append(r)
tabela = pd.DataFrame(linhas)
display(tabela.round(4))
tabela.to_csv(saida/'comparacao.csv',index=False)
for a,b in pares:
    print(a, textos[pos[a]], '\n', b, textos[pos[b]], '\n')
display(pd.DataFrame([{'Representação':n,'Linhas':x.shape[0],'Dimensões':x.shape[1]} for n,x in representacoes.items()]))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3610.79it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3672.77it/s]

,Par,BoW,TF-IDF,paraphrase-multilingual-MiniLM-L12-v2,distiluse-base-multilingual-cased-v2
0,M003 × M017,0.4009,0.3586,0.7939,0.8337
1,M008 × M022,0.5003,0.4920,0.9215,0.8562
2,M008 × M031,0.1356,0.0338,0.1984,0.1307


M003 Há um buraco enorme na Av. Brasil, a avenida principal do bairro Centro, em frente à padaria Pão Bom. Ele surgiu nesta semana e já fez dois carros desviarem para a contramão. 
 M017 O asfalto está todo esburacado na avenida principal do Centro, chamada Avenida Brasil, diante da padaria Pão Bom. É a cratera que apareceu esta semana e obriga os carros a invadir a pista contrária. 

M008 O posto de saúde São Lucas, na Rua das Acácias, está sem médico nesta semana. Fui à unidade pela manhã e não consegui consulta, pois não havia profissional para atender. 
 M022 Está faltando atendimento no PSF São Lucas da Rua das Acácias nesta semana. Pela manhã, fui procurar uma consulta e avisaram que nenhum médico estava disponível naquela unidade. 

M008 O posto de saúde São Lucas, na Rua das Acácias, está sem médico nesta semana. Fui à unidade pela manhã e não consegui consulta, pois não havia profissional para atender. 
 M031 A lâmpada está queimada na praça do bairro Bela Vista, junto ao banc

,Representação,Linhas,Dimensões
0,BoW,40,655
1,TF-IDF,40,655
2,paraphrase-multilingual-MiniLM-L12-v2,40,384
3,distiluse-base-multilingual-cased-v2,40,512


In [3]:
primeiro = MODELOS[0].split('/')[-1]
segundo = MODELOS[1].split('/')[-1]
display(Markdown(f"Nos pares do enunciado, o primeiro modelo obteve cossenos **{tabela[primeiro].iloc[0]:.3f}**, **{tabela[primeiro].iloc[1]:.3f}** e **{tabela[primeiro].iloc[2]:.3f}**. O segundo obteve **{tabela[segundo].iloc[0]:.3f}**, **{tabela[segundo].iloc[1]:.3f}** e **{tabela[segundo].iloc[2]:.3f}**. Esses valores devem ser interpretados junto aos textos: assunto semelhante não comprova que o local e o evento são os mesmos."))

Nos pares do enunciado, o primeiro modelo obteve cossenos **0.794**, **0.921** e **0.198**. O segundo obteve **0.834**, **0.856** e **0.131**. Esses valores devem ser interpretados junto aos textos: assunto semelhante não comprova que o local e o evento são os mesmos.

## Limitações e conclusão
BoW mede coincidência lexical e não reconhece sinônimos sem palavras compartilhadas. TF-IDF reduz a influência dos termos frequentes, mas continua dependente do vocabulário; um endereço raro pode dominar o score. Embeddings aproximam paráfrases, porém podem confundir ocorrências distintas do mesmo tema e produzir escalas diferentes entre modelos. Assim, busca semântica é adequada para recuperar candidatos, enquanto confirmar duplicatas exige verificar local, período e evento. Três pares ilustram diferenças, mas não constituem avaliação estatística da superioridade de um modelo.

## Espaço vetorial e categorias

In [4]:
e = representacoes[primeiro]
fig, axes = plt.subplots(1,2,figsize=(13,5))
for ax,metodo in zip(axes,['PCA','t-SNE']):
    coords=projetar(e,metodo)
    for categoria in sorted(set(base.categoria_oficial)):
        mask=(base.categoria_oficial==categoria).to_numpy()
        ax.scatter(coords[mask,0],coords[mask,1],label=categoria,alpha=.8)
    for xy,i in zip(coords,ids): ax.annotate(i,xy,fontsize=6)
    ax.set_title(metodo); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(saida/'espaco_vetorial.png',dpi=150); plt.show()
sim=e@e.T
mesma=[]; diferente=[]
for i in range(len(base)):
    for j in range(i+1,len(base)):
        (mesma if base.categoria_oficial.iloc[i]==base.categoria_oficial.iloc[j] else diferente).append(sim[i,j])
intra=float(np.mean(mesma)) if mesma else None
inter=float(np.mean(diferente)) if diferente else None
print('Cosseno médio dentro da categoria:',intra,'; entre categorias:',inter)
if intra is not None and inter is not None:
    display(Markdown(f'A média na categoria ({intra:.3f}) é '+('maior' if intra>inter else 'menor ou igual')+f' à média entre categorias ({inter:.3f}). Isso '+('sugere alinhamento temático parcial' if intra>inter else 'não sustenta separação temática por essa medida')+', mas não prova clusters separados. PCA perde variância e t-SNE distorce distâncias globais.'))
(saida/'representacoes_resumo.json').write_text(json.dumps({'pares':linhas,'intra_categoria':intra,'entre_categorias':inter},ensure_ascii=False,indent=2))

Cosseno médio dentro da categoria: 0.45196810364723206 ; entre categorias: 0.28096091747283936


A média na categoria (0.452) é maior à média entre categorias (0.281). Isso sugere alinhamento temático parcial, mas não prova clusters separados. PCA perde variância e t-SNE distorce distâncias globais.

836

## Referências
Materiais fornecidos: lab_representacoes.ipynb e laborat_ecommerce_produtos.ipynb. Implementação: https://www.sbert.net/ e https://scikit-learn.org/stable/modules/feature_extraction.html. O parâmetro stop_words="portuguese" do exemplo não é aceito pelo scikit-learn; por isso não é reproduzido.